# 🛒 Notebook 2 : ton premier agent qui agit

**Formation « Comprendre et développer des agents IA spécialisés »**, module 2.

## L'histoire

Tu travailles pour **MégaShop**, une boutique en ligne qui vend les essentiels du quotidien : smartphones, écouteurs, baskets, vêtements, sacs...
Chaque jour, les clients posent les mêmes questions :

- *« L'iPhone 16 rose est dispo ? »*
- *« Vous avez les Air Force 1 en 42 ? »*
- *« Je cherche un cadeau à moins de 60 €, vous conseillez quoi ? »*
- *« Je le reçois quand ? Je peux le renvoyer ? »*

Tu vas construire **Léa**, une conseillère de vente IA qui répond à ces questions **en consultant le vrai catalogue de la boutique** (stock, prix, avis clients) au lieu d'inventer.

⚠️ Ici, Léa **consulte** seulement (stock, prix, livraison, retours). Réserver, rembourser ou modifier une commande sont des actions **à risque** : elles exigent autorisation, contrôles métier et souvent une validation humaine (notebook 4).

## 🎯 Ce que tu vas apprendre

| Étape | Thème | Python pur | LangChain / LangGraph |
|---|---|:---:|:---:|
| 0 | Configuration | ✅ | |
| 1 | Le catalogue MégaShop | ✅ | |
| 2 | Pourquoi un LLM seul ne suffit pas | ✅ | |
| 3 | Créer des outils | ✅ | ✅ LangChain |
| 4 | Le tool calling | ✅ | ✅ LangChain |
| 5 | La boucle ReAct (l'agent) | ✅ | ✅ LangGraph |
| 6 | Sécuriser les outils avec Pydantic | ✅ | ✅ LangChain |
| 7 | Le scénario complet | ✅ | ✅ LangGraph |

**Principe du notebook** : à chaque étape, on comprend d'abord le mécanisme **en Python pur**, puis on voit **la même chose en quelques lignes avec un framework**.

## ▶️ Comment l'utiliser

- Exécute les cellules **dans l'ordre**, avec `Maj + Entrée`.
- Tout est dans ce notebook : il n'y a aucun fichier externe à ouvrir.
- Dans les traces de l'agent, tu verras 4 symboles :

| Symbole | Signification |
|---|---|
| 📝 | Un texte intermédiaire écrit par le LLM (rare) |
| 🔧 | L'agent appelle un outil |
| 📦 | L'agent reçoit le résultat de l'outil |
| 💬 | L'agent répond au client |

---
## Étape 0/7 · Configuration  `▱▱▱▱▱▱▱`

🎯 **Objectif** : choisir le modèle de langage (LLM) qui va animer Léa.

Tu as deux choix :
- **`"openai"`** (par défaut) : un modèle d'OpenAI via Internet. Il faut une clé `OPENAI_API_KEY` dans le fichier `.env`.
- **`"ollama"`** (option) : un modèle installé sur ton ordinateur. C'est gratuit et ça ne demande pas de clé, mais c'est moins fiable pour les agents.

👉 Change seulement la première ligne de la cellule ci-dessous.

In [1]:
FOURNISSEUR = "openai"          # "openai" (clé dans .env) ou "ollama" (local, gratuit)

import json, os
from datetime import date, timedelta
from dotenv import load_dotenv
from openai import OpenAI

AUJOURDHUI = date(2026, 10, 27)  # date fixe du scénario : mêmes résultats pour tout le monde

load_dotenv()
if FOURNISSEUR == "ollama":     # Ollama parle le même « langage » que l'API OpenAI
    MODELE, BASE_URL, CLE = "qwen2.5:7b", "http://localhost:11434/v1", "ollama"
else:
    MODELE, BASE_URL, CLE = "gpt-5-mini", None, os.getenv("OPENAI_API_KEY")
REGLAGES = {"temperature": 0} if FOURNISSEUR == "ollama" else {}   # réponses plus stables en local

client = OpenAI(base_url=BASE_URL, api_key=CLE)
print(f"✅ Prêt : {FOURNISSEUR} / {MODELE}")

✅ Prêt : openai / gpt-5-mini


On prépare aussi une petite fonction d'affichage, pour que les traces de l'agent soient faciles à lire.

In [2]:
ICONES = {"pensee": "📝 Note", "outil": "🔧 Appelle", "resultat": "📦 Reçoit", "reponse": "💬 Répond"}

def afficher(etape, texte):
    """Affiche une ligne de trace, par exemple : 🔧 Appelle │ verifier_stock(...)"""
    if etape == "reponse":
        print()                                  # une ligne vide avant la réponse finale
    print(f"{ICONES[etape]:<12} │ {texte}")

afficher("outil", "exemple d'affichage")

🔧 Appelle    │ exemple d'affichage


---
## Étape 1/7 · Le catalogue MégaShop  `▰▱▱▱▱▱▱`

🎯 **Objectif** : créer les données que Léa devra consulter.

Dans une vraie boutique, elles viendraient d'une base de données ou d'un logiciel e-commerce comme Shopify ou PrestaShop. Ici, on utilise de simples **dictionnaires Python** : c'est notre « mini base de données ».

On commence par les **informations générales** de la boutique :

In [3]:
BOUTIQUE = {
    "nom": "MégaShop",
    "slogan": "Les essentiels du quotidien, au meilleur prix",
    "livraison_gratuite_des": 100,           # en euros
    "prix_express": 9.90,                    # livraison express en 24 h
    "service_client": "du lundi au samedi, de 9 h à 18 h",
}
POLITIQUE_RETOUR = ("Retour gratuit sous 30 jours, article non utilisé et dans son emballage d'origine. "
                    "Remboursement sous 5 jours ouvrés après réception du colis.")
MOYENS_PAIEMENT = ["carte bancaire", "PayPal", "Apple Pay", "Google Pay", "paiement en 3 fois dès 150 €"]

CATALOGUE = {}                               # rempli dans les 3 cellules suivantes

Puis les **produits**, rangés en 3 rayons. Chaque produit a la même structure :

| Clé | Contenu |
|---|---|
| `prix`, `prix_barre` | Le prix, et l'ancien prix s'il est en promotion |
| `description`, `caracteristiques` | La fiche produit |
| `stock` | La quantité par couleur, ou par couleur **et** par taille pour les vêtements et les chaussures |
| `note`, `nombre_avis`, `avis` | Ce qu'en pensent les clients |

💡 Ce sont des cellules de **données** : pas besoin de tout lire, exécute-les simplement.

In [4]:
# 📱 Rayon HIGH-TECH
CATALOGUE["iphone 16"] = {
    "marque": "Apple", "categorie": "Téléphones", "prix": 899.00, "prix_barre": 969.00,
    "description": "Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,1 pouces", "photo": "48 MP", "garantie": "2 ans"},
    "stock": {"noir": 8, "blanc": 5, "rose": 0, "turquoise": 3},
    "note": 4.7, "nombre_avis": 328,
    "avis": [{"auteur": "Camille", "note": 5, "texte": "Très bon téléphone, rapide et excellent en photo."},
             {"auteur": "Thomas", "note": 4, "texte": "Très agréable à utiliser, mais le prix reste élevé."}],
}
CATALOGUE["samsung galaxy s25"] = {
    "marque": "Samsung", "categorie": "Téléphones", "prix": 849.00, "prix_barre": 899.00,
    "description": "Écran AMOLED 6,2 pouces, triple appareil photo et fonctions Galaxy AI. Parfait pour le multitâche.",
    "caracteristiques": {"stockage": "128 Go", "écran": "6,2 pouces", "photo": "50 MP", "garantie": "2 ans"},
    "stock": {"noir": 6, "bleu": 4, "argent": 2},
    "note": 4.6, "nombre_avis": 214,
    "avis": [{"auteur": "Sarah", "note": 5, "texte": "Écran magnifique et téléphone très fluide."},
             {"auteur": "Mehdi", "note": 4, "texte": "Très complet, notamment pour les photos."}],
}
CATALOGUE["écouteurs airpods 4"] = {
    "marque": "Apple", "categorie": "Audio", "prix": 149.00,
    "description": "Écouteurs sans fil avec audio spatial et boîtier de recharge USB-C.",
    "caracteristiques": {"autonomie": "5 h (30 h avec le boîtier)", "connexion": "Bluetooth", "micro": "oui"},
    "stock": {"blanc": 12},
    "note": 4.5, "nombre_avis": 192,
    "avis": [{"auteur": "Julie", "note": 5, "texte": "Connexion simple et son agréable."},
             {"auteur": "Alex", "note": 4, "texte": "Très pratiques dans les transports."}],
}
CATALOGUE["montre connectée xiaomi"] = {
    "marque": "Xiaomi", "categorie": "Montres", "prix": 49.99, "prix_barre": 59.99,
    "description": "Suivi d'activité, fréquence cardiaque, sommeil et notifications. Compatible Android et iPhone.",
    "caracteristiques": {"écran": "AMOLED", "autonomie": "14 jours", "étanchéité": "5 ATM"},
    "stock": {"noir": 9, "argent": 4},
    "note": 4.2, "nombre_avis": 231,
    "avis": [{"auteur": "Paul", "note": 5, "texte": "Bonne autonomie et suivi sportif pratique."},
             {"auteur": "Laura", "note": 3, "texte": "Bon rapport qualité-prix, application perfectible."}],
}

In [5]:
# 👕 Rayon MODE : le stock est donné par couleur PUIS par taille
CATALOGUE["baskets nike air force 1"] = {
    "marque": "Nike", "categorie": "Chaussures", "prix": 119.99, "prix_barre": 129.99,
    "description": "Baskets emblématiques en cuir, semelle épaisse, design intemporel. Unisexe.",
    "caracteristiques": {"matière": "cuir", "style": "lifestyle", "entretien": "chiffon humide"},
    "stock": {"blanc": {"39": 2, "40": 4, "41": 5, "42": 8, "43": 3, "44": 2, "45": 0},
              "noir":  {"39": 1, "40": 3, "41": 2, "42": 0, "43": 5, "44": 2, "45": 1}},
    "note": 4.6, "nombre_avis": 512,
    "avis": [{"auteur": "Lucas", "note": 5, "texte": "Très belles baskets, faciles à porter."},
             {"auteur": "Inès", "note": 4, "texte": "Modèle classique, mais un peu lourd."}],
}
CATALOGUE["hoodie adidas essentials"] = {
    "marque": "Adidas", "categorie": "Vêtements", "prix": 59.99, "prix_barre": 69.99,
    "description": "Sweat à capuche confortable avec poche kangourou et coupe décontractée.",
    "caracteristiques": {"matière": "coton et polyester", "coupe": "décontractée", "lavage": "30 °C"},
    "stock": {"noir": {"S": 3, "M": 5, "L": 4, "XL": 2}, "gris": {"S": 2, "M": 4, "L": 3, "XL": 1},
              "beige": {"S": 1, "M": 0, "L": 2, "XL": 2}},
    "note": 4.4, "nombre_avis": 146,
    "avis": [{"auteur": "Nicolas", "note": 5, "texte": "Très confortable et agréable à porter."},
             {"auteur": "Emma", "note": 3, "texte": "Belle qualité, mais taille un peu grand."}],
}
CATALOGUE["jean levis 501"] = {
    "marque": "Levi's", "categorie": "Vêtements", "prix": 89.99, "prix_barre": 99.99,
    "description": "Jean droit iconique à boutons, en denim résistant. Un classique du quotidien.",
    "caracteristiques": {"coupe": "droite", "matière": "denim", "lavage": "basse température"},
    "stock": {"bleu": {"36": 2, "38": 4, "40": 5, "42": 3, "44": 1}, "noir": {"36": 1, "38": 3, "40": 0, "42": 2, "44": 2}},
    "note": 4.3, "nombre_avis": 278,
    "avis": [{"auteur": "Antoine", "note": 5, "texte": "Une coupe classique qui dure dans le temps."},
             {"auteur": "Maya", "note": 4, "texte": "Très joli, mais vérifiez bien la taille."}],
}

In [6]:
# 🎒 Rayon QUOTIDIEN
CATALOGUE["sac à dos eastpak"] = {
    "marque": "Eastpak", "categorie": "Sacs", "prix": 54.99,
    "description": "Sac à dos compact de 24 litres, poche avant zippée et dos rembourré. Pour les cours ou le travail.",
    "caracteristiques": {"capacité": "24 L", "compartiment ordinateur": "non", "garantie": "30 ans"},
    "stock": {"noir": 10, "bleu": 4, "vert": 2},
    "note": 4.7, "nombre_avis": 346,
    "avis": [{"auteur": "Yanis", "note": 5, "texte": "Solide, léger et parfait pour les cours."},
             {"auteur": "Chloé", "note": 4, "texte": "Très pratique, mais pas de compartiment ordinateur."}],
}
CATALOGUE["gobelet stanley quencher"] = {
    "marque": "Stanley", "categorie": "Maison", "prix": 44.99,
    "description": "Gobelet isotherme de 1,18 L avec poignée, paille et couvercle. Garde le froid jusqu'à 11 h.",
    "caracteristiques": {"capacité": "1,18 L", "matière": "acier inoxydable", "lave-vaisselle": "oui"},
    "stock": {"rose": 3, "crème": 5, "noir": 7},
    "note": 4.5, "nombre_avis": 183,
    "avis": [{"auteur": "Anaïs", "note": 5, "texte": "Grande capacité et poignée très pratique."},
             {"auteur": "Romain", "note": 4, "texte": "Joli produit, mais assez encombrant."}],
}

Voici la **vitrine** de MégaShop. Un 🔥 signale les produits en promotion.

In [7]:
print(f"🛒 {BOUTIQUE['nom']} : {BOUTIQUE['slogan']}\n")
for nom, p in CATALOGUE.items():
    promo = "🔥" if p.get("prix_barre") else "  "
    print(f"{promo} {p['categorie']:<10} │ {nom:<25} │ {p['prix']:>7.2f} € │ ⭐ {p['note']} ({p['nombre_avis']} avis)")

🛒 MégaShop : Les essentiels du quotidien, au meilleur prix

🔥 Téléphones │ iphone 16                 │  899.00 € │ ⭐ 4.7 (328 avis)
🔥 Téléphones │ samsung galaxy s25        │  849.00 € │ ⭐ 4.6 (214 avis)
   Audio      │ écouteurs airpods 4       │  149.00 € │ ⭐ 4.5 (192 avis)
🔥 Montres    │ montre connectée xiaomi   │   49.99 € │ ⭐ 4.2 (231 avis)
🔥 Chaussures │ baskets nike air force 1  │  119.99 € │ ⭐ 4.6 (512 avis)
🔥 Vêtements  │ hoodie adidas essentials  │   59.99 € │ ⭐ 4.4 (146 avis)
🔥 Vêtements  │ jean levis 501            │   89.99 € │ ⭐ 4.3 (278 avis)
   Sacs       │ sac à dos eastpak         │   54.99 € │ ⭐ 4.7 (346 avis)
   Maison     │ gobelet stanley quencher  │   44.99 € │ ⭐ 4.5 (183 avis)


✅ **À retenir** : un agent n'est utile que s'il a accès à **tes** données (stock, prix, avis). Le LLM, lui, ne les connaît pas.

---
## Étape 2/7 · Pourquoi un LLM seul ne suffit pas  `▰▰▱▱▱▱▱`

🎯 **Objectif** : constater le problème qu'on va résoudre.

On pose une question de client au LLM, **sans lui donner accès au catalogue**.

In [8]:
question = "Bonjour, chez MégaShop, l'iPhone 16 rose est en stock ? Il coûte combien ?"

reponse = client.chat.completions.create(
    model=MODELE,
    messages=[{"role": "user", "content": question}],
    **REGLAGES,
)
afficher("reponse", reponse.choices[0].message.content)


💬 Répond     │ Bonjour — je n’ai pas accès aux stocks ni aux prix en temps réel des boutiques (dont MégaShop). Je ne peux donc pas vous dire directement si l’iPhone 16 rose est en stock ni son prix exact chez eux.

Je peux en revanche vous aider de plusieurs façons — dites-moi ce que vous préférez :
- Je peux vous expliquer rapidement comment vérifier le stock/prix sur le site MégaShop (ou via leur chat/téléphone).
- Si vous me donnez votre ville/pays ou l’adresse du magasin MégaShop concerné, je peux vous indiquer le meilleur moyen de les contacter (numéro ou page magasin).
- Si vous collez ici le lien produit MégaShop ou une capture d’écran, je peux vous aider à interpréter l’information.
- Si vous voulez juste une estimation indicative du prix (approximative), dites-moi le modèle exact (iPhone 16, 16 Plus, 16 Pro, capacité de stockage, neuf/reconditionné) et le pays — je vous donnerai une fourchette approximative.

Souhaitez-vous que je vous explique comment vérifier tout de suite 

🤔 **Que s'est-il passé ?** Le LLM **ne peut pas savoir** : il n'a jamais vu le stock de MégaShop, et il ignore que l'iPhone rose est en rupture. Soit il **invente** un prix ou une disponibilité, ce qui est dangereux pour un commerce, soit il répond qu'il ne sait pas, ce qui n'aide pas le client.

💡 **L'idée** : donner au LLM des **outils**, c'est-à-dire des fonctions Python qu'il peut **demander** d'appeler. C'est comme un vendeur qui va vérifier dans la réserve avant de répondre.

✅ **À retenir** : un LLM connaît le langage, pas ta boutique.

---
## Étape 3/7 · Créer des outils  `▰▰▰▱▱▱▱`

🎯 **Objectif** : écrire les fonctions que Léa pourra utiliser.

Un outil, c'est **une fonction Python normale** qui **renvoie du texte** avec `return`, et non `print`. C'est ce texte que le LLM lira.

On va créer 5 outils, un par type de question client :

| Outil | Question du client |
|---|---|
| `rechercher_produits` | « Vous avez quoi comme téléphones ? », « Un cadeau à moins de 60 € ? » |
| `fiche_produit` | « Il vaut le coup ? Que disent les clients ? » |
| `verifier_stock` | « Vous l'avez en noir, en taille 42 ? » |
| `delai_livraison` | « Je le reçois quand ? » |
| `infos_pratiques` | « Je peux le renvoyer ? Je peux payer avec PayPal ? » |

D'abord, deux petites fonctions pour **tolérer les approximations**. Le LLM écrira parfois « Air Force » au lieu de « baskets nike air force 1 », ou « bleue » au lieu de « bleu ».

In [9]:
import unicodedata

def normaliser(texte):
    """'  Écharpe ' -> 'echarpe' : minuscules, sans accents ni apostrophes."""
    texte = unicodedata.normalize("NFD", texte.lower()).encode("ascii", "ignore").decode()
    return texte.replace("'", " ").split()

def trouver(recherche, choix, strict=False):
    """Renvoie l'élément de `choix` qui partage le plus de mots avec `recherche`, ou None.
    strict=True : tous les mots demandés doivent correspondre (on ne devine pas)."""
    mots = [m for m in normaliser(recherche) if len(m) > 2]
    def score(c):        # « bleue » correspond à « bleu », « telephones » à « telephone »...
        return sum(any(m.startswith(k) or k.startswith(m) for k in normaliser(c) if len(k) > 2) for m in mots)
    meilleur = max(choix, key=score, default=None)
    if meilleur is None or score(meilleur) == 0 or (strict and score(meilleur) < len(mots)):
        return None
    return meilleur

print(trouver("Air Force", CATALOGUE), "|", trouver("Levi's", CATALOGUE), "|", trouver("bleue", ["noir", "bleu"]))

baskets nike air force 1 | jean levis 501 | bleu


⚠️ Être tolérant a un prix : le client demande des **Nike Air Max**, que MégaShop ne vend pas...

In [10]:
print("Tolérant :", trouver("Nike Air Max", CATALOGUE))               # ❌ choisit un autre produit
print("Strict   :", trouver("Nike Air Max", CATALOGUE, strict=True))  # ✅ ne devine pas

Tolérant : baskets nike air force 1
Strict   : None


👀 Le mode tolérant répond « Air Force 1 » : Léa risquerait d'annoncer un stock pour le **mauvais produit**.
Pour les produits, on utilise donc `strict=True` : en cas de doute, l'outil le dit, et Léa **demande une précision** au lieu de choisir au hasard.

### 🔧 Les outils « catalogue »

In [11]:
INCONNU = f"Produit introuvable ou ambigu : demande au client de préciser. Produits vendus : {', '.join(CATALOGUE)}."

def rechercher_produits(recherche: str = "", budget_max: float = 0) -> str:
    """Cherche des produits par mot-clé, marque ou catégorie (ex. téléphone, chaussures, promo), avec un budget maximum en euros (0 = sans limite)."""
    dans_budget = {n: p for n, p in CATALOGUE.items() if not budget_max or p["prix"] <= budget_max}
    trouves = {n: p for n, p in dans_budget.items()
               if trouver(recherche, [f"{n} {p['marque']} {p['categorie']} {'promo' if p.get('prix_barre') else ''}"])}
    entete = "" if trouves or not recherche else f"Rien ne correspond à « {recherche} ». Produits dans le budget :\n"
    lignes = []
    for nom, p in sorted((trouves or dans_budget).items(), key=lambda x: -x[1]["note"]):   # mieux notés d'abord
        promo = f" au lieu de {p['prix_barre']:.2f} €" if p.get("prix_barre") else ""
        lignes.append(f"- {nom} ({p['marque']}) : {p['prix']:.2f} €{promo}, note {p['note']}/5 ({p['nombre_avis']} avis)")
    return entete + ("\n".join(lignes) or "Aucun produit dans ce budget.")

print(rechercher_produits("téléphone"))
print(rechercher_produits("le mieux noté", budget_max=60))

- iphone 16 (Apple) : 899.00 € au lieu de 969.00 €, note 4.7/5 (328 avis)
- samsung galaxy s25 (Samsung) : 849.00 € au lieu de 899.00 €, note 4.6/5 (214 avis)
Rien ne correspond à « le mieux noté ». Produits dans le budget :
- sac à dos eastpak (Eastpak) : 54.99 €, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 44.99 €, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 59.99 € au lieu de 69.99 €, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 49.99 € au lieu de 59.99 €, note 4.2/5 (231 avis)


In [12]:
def fiche_produit(produit: str) -> str:
    """Donne la fiche complète d'un produit : prix, description, caractéristiques, couleurs en stock, note et avis clients."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    p = CATALOGUE[nom]
    prix = f"{p['prix']:.2f} €" + (f" au lieu de {p['prix_barre']:.2f} €" if p.get("prix_barre") else "")
    en_stock = [c for c, q in p["stock"].items() if (sum(q.values()) if isinstance(q, dict) else q) > 0]
    lignes = [f"{nom.upper()} ({p['marque']}) : {prix}", p["description"],
              "Caractéristiques : " + ", ".join(f"{k} : {v}" for k, v in p["caracteristiques"].items()),
              f"Couleurs en stock : {', '.join(en_stock) or 'aucune'}. Note : {p['note']}/5 ({p['nombre_avis']} avis)."]
    lignes += [f"  « {a['texte']} » ({a['auteur']}, {a['note']}/5)" for a in p["avis"]]
    return "\n".join(lignes)

print(fiche_produit("iphone"))

IPHONE 16 (Apple) : 899.00 € au lieu de 969.00 €
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Camille, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Thomas, 4/5)


`verifier_stock` est un peu plus longue, car elle gère **deux cas** : les produits avec une simple couleur (un iPhone noir) et les produits avec couleur **et** taille (des baskets blanches en 42).

In [13]:
def verifier_stock(produit: str, couleur: str, taille: str = "") -> str:
    """Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures."""
    nom = trouver(produit, CATALOGUE, strict=True)
    if nom is None:
        return INCONNU
    stock = CATALOGUE[nom]["stock"]
    teinte = trouver(couleur, stock)
    if teinte is None:
        return f"{nom} n'existe pas en {couleur}. Couleurs proposées : {', '.join(stock)}."
    if isinstance(stock[teinte], dict):                         # cas 2 : article avec tailles
        quantite = stock[teinte].get(taille.upper(), 0)
        dispo = [t for t, q in stock[teinte].items() if q > 0]
        if quantite == 0:
            return f"{nom} {teinte} : taille '{taille}' indisponible. Tailles en stock : {', '.join(dispo) or 'aucune'}."
        return f"{nom} {teinte} taille {taille.upper()} : {quantite} en stock."
    if stock[teinte] > 0:                                       # cas 1 : couleur seule
        return f"{nom} {teinte} : {stock[teinte]} en stock."
    autres = [c for c, q in stock.items() if q > 0]
    return f"{nom} {teinte} : rupture de stock. Couleurs disponibles : {', '.join(autres) or 'aucune'}."

print(verifier_stock("iPhone 16", "rose"))
print(verifier_stock("Air Force", "blanches", "42"))
print(verifier_stock("hoodie", "beige", "M"))

iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
baskets nike air force 1 blanc taille 42 : 8 en stock.
hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.


### 🔧 Les outils « service client »

In [14]:
JOURS = ["lundi", "mardi", "mercredi", "jeudi", "vendredi", "samedi", "dimanche"]

def dans_jours_ouvres(n):
    """Date dans n jours ouvrés (on saute samedi et dimanche), ex. 'jeudi 29/10'."""
    jour = AUJOURDHUI
    while n > 0:
        jour += timedelta(days=1)
        if jour.weekday() < 5:
            n -= 1
    return f"{JOURS[jour.weekday()]} {jour:%d/%m}"

def delai_livraison(code_postal: str) -> str:
    """Estime les dates de livraison standard et express à partir du code postal du client."""
    standard = 2 if code_postal[:2] in ("75", "92", "93", "94") else 3      # Paris et proche banlieue : plus rapide
    return (f"Standard : {dans_jours_ouvres(standard)} (offerte dès {BOUTIQUE['livraison_gratuite_des']} €). "
            f"Express : {dans_jours_ouvres(1)} ({BOUTIQUE['prix_express']:.2f} €).")

In [15]:
def infos_pratiques() -> str:
    """Donne les conditions de retour et de remboursement, les moyens de paiement et les horaires du service client."""
    return (f"Retours : {POLITIQUE_RETOUR} Paiement : {', '.join(MOYENS_PAIEMENT)}. "
            f"Service client : {BOUTIQUE['service_client']}.")

print(delai_livraison("69003"))
print(infos_pratiques())

Standard : vendredi 30/10 (offerte dès 100 €). Express : mercredi 28/10 (9.90 €).
Retours : Retour gratuit sous 30 jours, article non utilisé et dans son emballage d'origine. Remboursement sous 5 jours ouvrés après réception du colis. Paiement : carte bancaire, PayPal, Apple Pay, Google Pay, paiement en 3 fois dès 150 €. Service client : du lundi au samedi, de 9 h à 18 h.


### 🧩 Décrire l'outil au LLM (Python pur)

Le LLM ne lit pas ton code Python. Il faut lui envoyer une **fiche descriptive** de chaque outil au format JSON : son nom, à quoi il sert et ses paramètres.

⚠️ Cette description est **un prompt** : plus elle est claire, mieux le LLM choisira le bon outil. Voici la fiche de `verifier_stock`, écrite à la main :

In [16]:
schema_stock = {
    "type": "function",
    "function": {
        "name": "verifier_stock",
        "description": "Vérifie le stock d'un produit dans une couleur, et dans une taille pour les vêtements et chaussures.",
        "parameters": {
            "type": "object",
            "properties": {
                "produit": {"type": "string", "description": "Nom du produit, ex. iphone 16"},
                "couleur": {"type": "string", "description": "Couleur demandée, ex. noir"},
                "taille":  {"type": "string", "description": "Taille, ex. 42 ou M (vide si sans taille)"},
            },
            "required": ["produit", "couleur"],          # la taille est facultative
        },
    },
}

Écrire 5 fiches à la main serait long. La fonction `decrire()` les **fabrique automatiquement** en lisant la docstring et la signature de chaque fonction : le nom des paramètres, leur type et s'ils sont obligatoires.

In [17]:
import inspect
TYPES_JSON = {str: "string", float: "number", int: "integer", bool: "boolean"}

def decrire(fonction):
    """Fabrique la fiche JSON d'un outil à partir de sa docstring et de sa signature."""
    params = inspect.signature(fonction).parameters
    proprietes = {nom: {"type": TYPES_JSON[p.annotation]} for nom, p in params.items()}
    obligatoires = [nom for nom, p in params.items() if p.default is inspect.Parameter.empty]
    return {"type": "function", "function": {"name": fonction.__name__, "description": fonction.__doc__,
            "parameters": {"type": "object", "properties": proprietes, "required": obligatoires}}}

FONCTIONS = [rechercher_produits, fiche_produit, verifier_stock, delai_livraison, infos_pratiques]
SCHEMAS = [decrire(f) for f in FONCTIONS]
OUTILS = {f.__name__: f for f in FONCTIONS}            # nom -> fonction, pour les exécuter
print(json.dumps(SCHEMAS[3], ensure_ascii=False, indent=1))

{
 "type": "function",
 "function": {
  "name": "delai_livraison",
  "description": "Estime les dates de livraison standard et express à partir du code postal du client.",
  "parameters": {
   "type": "object",
   "properties": {
    "code_postal": {
     "type": "string"
    }
   },
   "required": [
    "code_postal"
   ]
  }
 }
}


💡 `decrire()` est un **générateur pédagogique** : il ne gère que des paramètres simples, sans exemples ni valeurs autorisées. En production, on écrit un schéma explicite, souvent avec **Pydantic** (étape 6).

### 🔁 La même chose avec LangChain

LangChain fait exactement ce que fait `decrire()` : il lit **le nom, la docstring et les annotations de type** de ta fonction.

In [18]:
from langchain_core.tools import tool

outils_lc = [tool(f) for f in FONCTIONS]

for o in outils_lc:
    print(f"🔧 {o.name:<20} │ paramètres : {list(o.args)}")

🔧 rechercher_produits  │ paramètres : ['recherche', 'budget_max']
🔧 fiche_produit        │ paramètres : ['produit']
🔧 verifier_stock       │ paramètres : ['produit', 'couleur', 'taille']
🔧 delai_livraison      │ paramètres : ['code_postal']
🔧 infos_pratiques      │ paramètres : []


✅ **À retenir**
- Un outil = **une fonction Python qui renvoie du texte** + **une description** pour le LLM.
- En Python pur, tu écris la description en JSON. LangChain la génère à partir de la docstring.
- Une bonne docstring, c'est un bon outil.

---
## Étape 4/7 · Le tool calling  `▰▰▰▰▱▱▱`

🎯 **Objectif** : voir comment le LLM **demande** à utiliser un outil.

Le point clé : **le LLM n'exécute jamais rien lui-même**. Il répond : *« Appelle `verifier_stock` avec `produit=air force`, `couleur=blanc` et `taille=42` »*, et c'est **ton code** qui exécute la fonction.

```
 Ton code ──(question + fiches des outils)──▶ LLM
 Ton code ◀──── « appelle verifier_stock » ─── LLM
 Ton code : exécute verifier_stock(...)
 Ton code ──────── (résultat de l'outil) ───▶ LLM
 Ton code ◀────────── réponse au client ────── LLM
```

On commence par le **prompt système**, la « fiche de poste » de Léa. Il indique **quel outil utiliser pour quelle question**, ce qui aide beaucoup les petits modèles locaux.

In [19]:
SYSTEME = f"""Tu es Léa, conseillère de vente de la boutique en ligne {BOUTIQUE['nom']} ({BOUTIQUE['slogan']}).
Nous sommes le {JOURS[AUJOURDHUI.weekday()]} {AUJOURDHUI:%d/%m/%Y}.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (code postal, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux."""
print(SYSTEME)

Tu es Léa, conseillère de vente de la boutique en ligne MégaShop (Les essentiels du quotidien, au meilleur prix).
Nous sommes le mardi 27/10/2026.
N'invente jamais : chaque information doit venir d'un outil.
Tu consultes seulement : tu ne peux ni réserver, ni commander, ni rembourser.
- Disponibilité d'une couleur ou d'une taille : verifier_stock.
- Date de livraison : delai_livraison. Retours, paiement, horaires : infos_pratiques.
- Conseil ou recherche de produits : rechercher_produits, puis fiche_produit pour les avis.
Si le client pose plusieurs questions, appelle tous les outils nécessaires avant de répondre.
Ne dis jamais « je vais vérifier » : vérifie tout de suite avec l'outil.
Si une information du client manque ou est invalide (code postal, taille...), demande-la : ne la devine jamais.
Réponds en français, en 4 phrases maximum, avec un ton chaleureux.


### 4a. Le LLM demande un outil (Python pur)

In [20]:
messages = [{"role": "system", "content": SYSTEME},
            {"role": "user", "content": "Vous avez les Nike Air Force 1 blanches en 42 ?"}]

reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
demande = reponse.choices[0].message

print("Texte du LLM :", repr(demande.content))          # souvent vide : il préfère demander un outil
for appel in demande.tool_calls or []:
    afficher("outil", f"{appel.function.name}({appel.function.arguments})")

Texte du LLM : None
🔧 Appelle    │ verifier_stock({"produit":"Nike Air Force 1","couleur":"blanc","taille":"42"})


👀 Le LLM n'a pas répondu au client : il a renvoyé **une demande d'appel**, avec le nom de l'outil et ses arguments en JSON.

### 4b. Ton code exécute l'outil et renvoie le résultat (Python pur)

In [21]:
def executer(nom, arguments):
    """Exécute l'outil demandé par le LLM (on la renforcera à l'étape 6)."""
    return OUTILS[nom](**arguments)

appel = demande.tool_calls[0]
arguments = json.loads(appel.function.arguments)        # texte JSON -> dictionnaire Python
resultat = executer(appel.function.name, arguments)     # on exécute la vraie fonction
afficher("resultat", resultat)

messages.append(demande)                                # on garde la demande du LLM...
messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})  # ...et le résultat

final = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
afficher("reponse", final.choices[0].message.content)

📦 Reçoit     │ baskets nike air force 1 blanc taille 42 : 8 en stock.

💬 Répond     │ Bonne nouvelle — nous avons 8 paires de Nike Air Force 1 blanches en 42 en stock. Voulez-vous que je vérifie le délai de livraison pour votre code postal ?


### 🔁 La même chose avec LangChain

`bind_tools()` « attache » les outils au LLM. Les arguments arrivent déjà convertis en dictionnaire Python.

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model=MODELE, base_url=BASE_URL, api_key=CLE, **REGLAGES)   # marche aussi avec Ollama
llm_avec_outils = llm.bind_tools(outils_lc)

demande_lc = llm_avec_outils.invoke([("system", SYSTEME), ("user", "Vous avez les Nike Air Force 1 blanches en 42 ?")])
for appel in demande_lc.tool_calls:
    afficher("outil", f"{appel['name']}({appel['args']})")

🔧 Appelle    │ verifier_stock({'produit': 'Nike Air Force 1', 'couleur': 'blanche', 'taille': '42'})


✅ **À retenir**
- Le LLM **choisit** l'outil et ses arguments, mais c'est **ton code** qui l'exécute. C'est toi qui gardes le contrôle.
- Le résultat est renvoyé au LLM dans un message de rôle `"tool"`, puis le LLM rédige la réponse au client.
- 💡 Pour un nouveau projet, OpenAI recommande son API *Responses*. On garde ici *Chat Completions*, dont la boucle est très lisible.

---
## Étape 5/7 · La boucle ReAct : de l'outil à l'agent  `▰▰▰▰▰▱▱`

🎯 **Objectif** : laisser le LLM enchaîner **plusieurs outils** tout seul, jusqu'à avoir de quoi répondre.

À l'étape 4, on a fait **un seul aller-retour à la main**. Mais pour conseiller un cadeau, Léa doit d'abord **chercher** des produits, puis **lire les avis**, puis **répondre**. **ReAct** (*Reason + Act*) consiste à répéter le cycle suivant :

```
   ┌──▶ 🧠 RÉFLÉCHIR : « De quoi ai-je besoin ? »
   │         │
   │    🔧 AGIR : appeler un outil
   │         │
   └─── 📦 OBSERVER : lire le résultat
             │
        (plus besoin d'outil ?)
             ▼
        💬 RÉPONDRE au client
```

⚠️ La « réflexion » se passe **dans** le modèle : on ne la voit pas. Ce qu'on peut observer et vérifier, ce sont ses **décisions** : quel outil, quels arguments, quel résultat. Si le LLM écrit un petit texte en plus, on l'affiche en 📝, sans le prendre pour son vrai raisonnement.

C'est **le LLM qui décide** quand s'arrêter. C'est ce qui fait la différence entre un simple script et un **agent**.

### En Python pur : une simple boucle `for`

In [23]:
def agent_python(question, max_tours=6):
    messages = [{"role": "system", "content": SYSTEME}, {"role": "user", "content": question}]
    for _ in range(max_tours):                       # garde-fou : jamais de boucle infinie
        reponse = client.chat.completions.create(model=MODELE, messages=messages, tools=SCHEMAS, **REGLAGES)
        msg = reponse.choices[0].message
        messages.append(msg)
        if not msg.tool_calls:                       # plus d'outil demandé -> réponse finale
            return afficher("reponse", msg.content)
        if msg.content:
            afficher("pensee", msg.content)          # 📝 texte éventuel (pas son raisonnement interne)
        for appel in msg.tool_calls:
            arguments = json.loads(appel.function.arguments or "{}")
            afficher("outil", f"{appel.function.name}({arguments})")        # 🔧 AGIR
            resultat = executer(appel.function.name, arguments)
            afficher("resultat", resultat)                                  # 📦 OBSERVER
            messages.append({"role": "tool", "tool_call_id": appel.id, "content": resultat})
    afficher("reponse", "Désolée, je n'ai pas réussi à répondre. Un conseiller va vous recontacter.")

In [24]:
agent_python("Je cherche un cadeau à moins de 60 €. Qu'est-ce qui est le mieux noté chez vous ?")

🔧 Appelle    │ rechercher_produits({'recherche': '', 'budget_max': 60})
📦 Reçoit     │ - sac à dos eastpak (Eastpak) : 54.99 €, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 44.99 €, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 59.99 € au lieu de 69.99 €, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 49.99 € au lieu de 59.99 €, note 4.2/5 (231 avis)

💬 Répond     │ Bonjour ! Le mieux noté sous 60 € chez nous est le sac à dos Eastpak : 54,99 €, note 4,7/5 (346 avis). Je peux afficher la fiche détaillée et les avis, ou vous proposer d'autres options (gobelet Stanley 44,99 €, 4,5/5 ; montre Xiaomi 49,99 €, 4,2/5). Que préférez-vous ?


### 🔁 La même chose avec LangGraph

LangGraph sert à construire des agents sous forme de **graphe**. Notre boucle devient deux **nœuds** (des cases) reliés par des **flèches** :

```
  [début] ──▶ (llm) ──outil demandé ?── oui ──▶ (outils)
                ▲  │                              │
                │  └── non ──▶ [fin]              │
                └─────────────────────────────────┘
```

| Élément LangGraph | Rôle | Équivalent dans `agent_python` |
|---|---|---|
| `MessagesState` | La mémoire partagée : la liste des messages | la liste `messages` |
| nœud `llm` | Appelle le LLM avec les outils | `client.chat.completions.create(...)` |
| nœud `outils` (`ToolNode`) | Exécute les outils demandés | la boucle `for appel in ...` |
| `tools_condition` | Décide : vers les outils, ou fin | `if not msg.tool_calls` |

In [25]:
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, tools_condition

def construire_agent(outils):
    llm_outils = llm.bind_tools(outils)
    def noeud_llm(etat):                              # 🧠 le LLM lit l'historique et décide
        return {"messages": [llm_outils.invoke([("system", SYSTEME)] + etat["messages"])]}
    graphe = StateGraph(MessagesState)
    graphe.add_node("llm", noeud_llm)
    graphe.add_node("outils", ToolNode(outils))       # 🔧 📦 exécute les outils demandés
    graphe.add_edge(START, "llm")
    graphe.add_conditional_edges("llm", tools_condition, {"tools": "outils", END: END})
    graphe.add_edge("outils", "llm")                  # après les outils, retour au LLM
    return graphe.compile()

agent_lg = construire_agent(outils_lc)
print("✅ Agent LangGraph prêt :", list(agent_lg.get_graph().nodes))

✅ Agent LangGraph prêt : ['__start__', 'llm', 'outils', '__end__']


Comme pour la version Python, une petite fonction transforme l'historique des messages en traces lisibles.

In [26]:
def afficher_messages(messages):
    """Traduit l'historique LangGraph en traces 🧠 🔧 📦 💬."""
    for m in messages:
        if m.type == "ai" and m.tool_calls:
            if m.content:
                afficher("pensee", m.content)
            for appel in m.tool_calls:
                afficher("outil", f"{appel['name']}({appel['args']})")
        elif m.type == "tool":
            afficher("resultat", m.content)
        elif m.type == "ai":
            afficher("reponse", m.content)

In [27]:
resultat = agent_lg.invoke({"messages": [("user", "Je cherche un cadeau à moins de 60 €. Qu'est-ce qui est le mieux noté chez vous ?")]})
afficher_messages(resultat["messages"])

🔧 Appelle    │ rechercher_produits({'recherche': '', 'budget_max': 60})
📦 Reçoit     │ - sac à dos eastpak (Eastpak) : 54.99 €, note 4.7/5 (346 avis)
- gobelet stanley quencher (Stanley) : 44.99 €, note 4.5/5 (183 avis)
- hoodie adidas essentials (Adidas) : 59.99 € au lieu de 69.99 €, note 4.4/5 (146 avis)
- montre connectée xiaomi (Xiaomi) : 49.99 € au lieu de 59.99 €, note 4.2/5 (231 avis)
🔧 Appelle    │ fiche_produit({'produit': 'sac à dos eastpak (Eastpak)'})
📦 Reçoit     │ Produit introuvable ou ambigu : demande au client de préciser. Produits vendus : iphone 16, samsung galaxy s25, écouteurs airpods 4, montre connectée xiaomi, baskets nike air force 1, hoodie adidas essentials, jean levis 501, sac à dos eastpak, gobelet stanley quencher.

💬 Répond     │ Le mieux noté à moins de 60 € est le sac à dos Eastpak à 54,99 € (note 4,7/5 sur 346 avis).  
Autres options bien notées : le gobelet Stanley Quencher 44,99 € (4,5/5) et la montre connectée Xiaomi 49,99 € (4,2/5).  
Souhaitez-vous

✅ **À retenir**
- Un **agent**, c'est un LLM + des outils + **une boucle**. Le LLM décide de l'étape suivante.
- Il faut toujours un **garde-fou** (`max_tours`) pour éviter une boucle sans fin. Dans LangGraph, c'est `recursion_limit`.
- LangGraph rend la boucle **visible** sous forme de graphe. Tu peux ensuite ajouter des nœuds, par exemple une validation humaine avant un remboursement.
- 💡 Raccourci : `create_agent` (dans le package `langchain`) construit ce même graphe en une seule ligne.

---
## Étape 6/7 · Sécuriser les outils avec Pydantic  `▰▰▰▰▰▰▱`

🎯 **Objectif** : vérifier les arguments envoyés par le LLM **avant** d'exécuter un outil.

Le LLM n'est pas infaillible. Il peut envoyer `code_postal="Lyon"` au lieu de `"69003"`, ou `taille=42` (un nombre) au lieu de `"42"` (un texte). Entre la demande du LLM et l'exécution, **ton programme** a le dernier mot :

```
Client
  │ question
  ▼
LLM ── demande un outil + arguments ──► Programme
 ▲                                        │
 │                                        ├─ valide le format
 │ résultat structuré                     ├─ applique les règles métier
 └────────────────────────────────────────└─ exécute ou refuse
```

Regardons ce qui se passe sans validation :

In [28]:
print(delai_livraison("Lyon"))                       # ❌ pas d'erreur... mais une date fausse
try:
    verifier_stock("air force", "blanc", 42)         # ❌ un nombre au lieu d'un texte
except Exception as erreur:
    print("💥 Plantage :", repr(erreur))

Standard : vendredi 30/10 (offerte dès 100 €). Express : mercredi 28/10 (9.90 €).
💥 Plantage : AttributeError("'int' object has no attribute 'upper'")


Deux problèmes : une **réponse fausse en silence**, et un **plantage** qui casserait la boucle de l'agent.

💡 **Pydantic** est la bibliothèque Python de référence pour **valider des données**. On décrit la forme attendue des arguments dans une classe, et Pydantic :
- ✅ **corrige** ce qui peut l'être, par exemple `42` devient `"42"` ;
- ❌ **refuse** ce qui est invalide, avec un message clair, par exemple « le code postal doit faire 5 chiffres ».

In [29]:
from pydantic import BaseModel, ConfigDict, Field, ValidationError

class ArgsLivraison(BaseModel):
    code_postal: str = Field(pattern=r"^\d{5}$", description="Code postal français à 5 chiffres, ex. 69003")

class ArgsStock(BaseModel):
    model_config = ConfigDict(coerce_numbers_to_str=True)       # 42 -> "42"
    produit: str = Field(description="Nom du produit, ex. iphone 16")
    couleur: str = Field(description="Couleur demandée, ex. noir")
    taille: str = Field("", description="Taille, ex. 42 ou M (vide si sans taille)")

print("✅", ArgsStock(produit="air force", couleur="blanc", taille=42))
try:
    ArgsLivraison(code_postal="Lyon")
except ValidationError as erreur:
    print("❌", erreur.errors()[0]["msg"])

✅ produit='air force' couleur='blanc' taille='42'
❌ String should match pattern '^\d{5}$'


### 🔒 Brancher la validation dans l'agent (Python pur)

On remplace `executer()` par une version qui **valide d'abord**. L'astuce : en cas d'erreur, on ne plante pas, on **renvoie l'erreur au LLM** comme résultat de l'outil. Grâce à la boucle ReAct, il peut alors **se corriger** ou **demander l'information au client**.

Comme `agent_python()` appelle `executer()`, il profite tout de suite de la nouvelle version.

In [30]:
VALIDATEURS = {"delai_livraison": ArgsLivraison, "verifier_stock": ArgsStock}

def executer(nom, arguments):
    """Valide les arguments avec Pydantic, puis exécute l'outil. En cas d'erreur, prévient le LLM."""
    if nom not in OUTILS:
        return f"Outil inconnu : {nom}."
    if nom in VALIDATEURS:
        try:
            arguments = VALIDATEURS[nom](**arguments).model_dump()
        except ValidationError as erreur:
            detail = erreur.errors()[0]
            return f"Arguments invalides ({detail['loc'][0]} : {detail['msg']}). Corrige-les ou demande l'information au client."
    try:
        return OUTILS[nom](**arguments)
    except Exception as erreur:                         # panne, délai dépassé...
        return f"Erreur technique ({erreur}). N'invente rien : propose au client de réessayer plus tard."

print(executer("delai_livraison", {"code_postal": "Lyon"}))
print(executer("verifier_stock", {"produit": "air force", "couleur": "blanc", "taille": 42}))

Arguments invalides (code_postal : String should match pattern '^\d{5}$'). Corrige-les ou demande l'information au client.
baskets nike air force 1 blanc taille 42 : 8 en stock.


In [31]:
agent_python("Je voudrais le sac Eastpak noir, je le reçois quand ? Mon code postal : 6903.")

🔧 Appelle    │ rechercher_produits({'recherche': 'sac Eastpak noir', 'budget_max': 0})
📦 Reçoit     │ - sac à dos eastpak (Eastpak) : 54.99 €, note 4.7/5 (346 avis)
🔧 Appelle    │ verifier_stock({'produit': 'sac à dos eastpak', 'couleur': 'noir'})
📦 Reçoit     │ sac à dos eastpak noir : 10 en stock.
🔧 Appelle    │ delai_livraison({'code_postal': '6903'})
📦 Reçoit     │ Arguments invalides (code_postal : String should match pattern '^\d{5}$'). Corrige-les ou demande l'information au client.

💬 Répond     │ Bonjour — bonne nouvelle : le sac à dos Eastpak noir est en stock (10 exemplaires).  
Le code postal que vous avez indiqué ("6903") n’est pas valide, il doit comporter 5 chiffres.  
Pouvez-vous me communiquer votre code postal à 5 chiffres pour que j’estime la date de livraison ?


👀 Le client a fait une **faute de frappe** : son code postal n'a que 4 chiffres. Pydantic refuse l'argument, et Léa reçoit l'erreur comme résultat de l'outil. Au lieu d'inventer une date, elle **demande au client de vérifier** son code postal.

### 🔁 La même chose avec LangChain

On donne le modèle Pydantic à `tool()` avec `args_schema`. LangChain s'en sert pour **fabriquer la fiche JSON** (avec les descriptions des champs) et pour **valider les arguments**. Dans LangGraph, `ToolNode` renvoie automatiquement l'erreur au LLM, comme notre `executer()`.

In [32]:
outils_valides = [tool(rechercher_produits), tool(fiche_produit), tool(infos_pratiques),
                  tool(verifier_stock, args_schema=ArgsStock),
                  tool(delai_livraison, args_schema=ArgsLivraison)]

agent_lg = construire_agent(outils_valides)                      # même graphe, outils sécurisés
afficher_messages(agent_lg.invoke({"messages": [("user", "Je voudrais le sac Eastpak noir, je le reçois quand ? Mon code postal : 6903.")]})["messages"])

🔧 Appelle    │ verifier_stock({'produit': 'sac Eastpak', 'couleur': 'noir', 'taille': ''})
📦 Reçoit     │ sac à dos eastpak noir : 10 en stock.

💬 Répond     │ Bonjour, c’est Léa de MégaShop — le sac Eastpak noir est disponible, il y en a 10 en stock. Le code postal que vous avez indiqué (6903) est invalide : il faut un code postal français à 5 chiffres. Pouvez-vous me donner votre code postal correct (5 chiffres) pour que je vous indique la date de livraison estimée ?


### 🧯 Et quand c'est le système qui casse ?

Un agent rencontre aussi des erreurs qui ne viennent pas des arguments :

| Problème | Ce que fait le programme |
|---|---|
| Outil inconnu | répond « outil inconnu » au LLM |
| Arguments JSON illisibles | renvoie l'erreur au LLM, ou s'arrête proprement |
| Catalogue indisponible, délai dépassé | renvoie « erreur technique », sans planter |
| Résultat vide ou incohérent | contrôle métier avant de répondre |

Simulons une **panne du catalogue** :

In [33]:
def stock_en_panne(**arguments):
    raise TimeoutError("le catalogue ne répond pas")

OUTILS["verifier_stock"] = stock_en_panne        # 💥 on simule la panne
agent_python("Vous avez l'iPhone 16 en noir ?")
OUTILS["verifier_stock"] = verifier_stock        # ✅ on répare

🔧 Appelle    │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'noir'})
📦 Reçoit     │ Erreur technique (le catalogue ne répond pas). N'invente rien : propose au client de réessayer plus tard.

💬 Répond     │ Désolée, le catalogue ne répond pas en ce moment et je ne peux pas vérifier la disponibilité de l’iPhone 16 en noir. Pouvez-vous réessayer dans quelques minutes ou me redemander un peu plus tard ? Si vous préférez, dites-moi si une autre couleur ou le modèle Pro vous intéresse et je retenterai dès que le service sera rétabli. — Léa, MégaShop


👀 Léa n'invente pas de stock et ne réessaie pas sans fin (`max_tours` l'en empêche). **Une boucle ReAct ne remplace pas les garde-fous d'un logiciel.**

✅ **À retenir**
- Les arguments venant d'un LLM sont des **données non fiables**, comme un formulaire rempli par un inconnu. **On les valide toujours.**
- On **renvoie l'erreur au LLM** au lieu de planter : la boucle ReAct lui permet de se corriger.
- ⚠️ Mais « se corriger » ne doit pas vouloir dire **deviner** : sans consigne, un LLM peut transformer `6903` en `06903`. C'est pourquoi le prompt système dit : *« demande-la : ne la devine jamais »*.
- Bonus : la classe Pydantic sert **à la fois** à valider et à décrire l'outil au LLM.

Pydantic ne protège qu'**un** des trois niveaux :

| Niveau | Exemple | Protection |
|---|---|---|
| Format | `6903` au lieu de 5 chiffres | Pydantic |
| Métier | taille 46 inexistante pour ce produit | l'outil, les règles métier |
| Autorisation | réserver ou rembourser | règles de l'application + validation humaine |

`42 → "42"` est une conversion pratique, mais elle ne dit pas si la taille existe, ni si la demande est légitime.

---
## Étape 7/7 · Le scénario complet  `▰▰▰▰▰▰▰`

🎯 **Objectif** : un vrai message client, avec **plusieurs questions en une**.

Observe les traces : Léa doit **décider seule** quels outils utiliser. Le piège : l'iPhone rose est en rupture, et la livraison standard vers Lyon arrive trop tard. Saura-t-elle proposer une autre couleur et l'option express ?

In [34]:
CLIENT = ("Bonjour ! L'iPhone 16 rose est dispo ? Sinon, quelles couleurs vous reste-t-il ? "
          "J'habite à Lyon (69003) : je peux l'avoir avant vendredi ? Et je peux le renvoyer s'il ne me plaît pas ?")

print("════ Version Python pur ════")
agent_python(CLIENT)

════ Version Python pur ════
🔧 Appelle    │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose'})
📦 Reçoit     │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
🔧 Appelle    │ fiche_produit({'produit': 'iPhone 16'})
📦 Reçoit     │ IPHONE 16 (Apple) : 899.00 € au lieu de 969.00 €
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Camille, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Thomas, 4/5)
🔧 Appelle    │ delai_livraison({'code_postal': '69003'})
📦 Reçoit     │ Standard : vendredi 30/10 (offerte dès 100 €). Express : mercredi 28/10 (9.90 €).
🔧 Appelle    │ infos_pratiques({})
📦 Reçoit     │ Retours : Retour gratuit sous 30 jours, article non utilisé et dans son

In [35]:
print("════ Version LangGraph ════")
afficher_messages(agent_lg.invoke({"messages": [("user", CLIENT)]})["messages"])

════ Version LangGraph ════
🔧 Appelle    │ verifier_stock({'produit': 'iPhone 16', 'couleur': 'rose', 'taille': ''})
🔧 Appelle    │ fiche_produit({'produit': 'iPhone 16'})
🔧 Appelle    │ delai_livraison({'code_postal': '69003'})
🔧 Appelle    │ infos_pratiques({})
📦 Reçoit     │ iphone 16 rose : rupture de stock. Couleurs disponibles : noir, blanc, turquoise.
📦 Reçoit     │ IPHONE 16 (Apple) : 899.00 € au lieu de 969.00 €
Écran 6,1 pouces, puce A18, appareil photo 48 MP et USB-C. Idéal pour la photo et les réseaux sociaux.
Caractéristiques : stockage : 128 Go, écran : 6,1 pouces, photo : 48 MP, garantie : 2 ans
Couleurs en stock : noir, blanc, turquoise. Note : 4.7/5 (328 avis).
  « Très bon téléphone, rapide et excellent en photo. » (Camille, 5/5)
  « Très agréable à utiliser, mais le prix reste élevé. » (Thomas, 4/5)
📦 Reçoit     │ Standard : vendredi 30/10 (offerte dès 100 €). Express : mercredi 28/10 (9.90 €).
📦 Reçoit     │ Retours : Retour gratuit sous 30 jours, article non utilis

👀 Dans la trace LangGraph, plusieurs 🔧 arrivent **d'un coup** avant les 📦 : le LLM peut demander plusieurs outils **dans le même tour** (stock, livraison, retours). Le programme les exécute tous, puis lui renvoie tous les résultats.

💡 **Python pur et LangGraph ne donnent pas toujours exactement la même réponse.** Le LLM n'est pas déterministe. Et si tu essaies `FOURNISSEUR = "ollama"` à l'étape 0, tu verras qu'un petit modèle local comme qwen2.5:7b **oublie parfois un outil**, par exemple la livraison : il répond alors sans avoir vérifié. C'est une vraie leçon : **la fiabilité d'un agent dépend du modèle**, mais aussi de la clarté des descriptions d'outils, du prompt, des données et du traitement des erreurs.

### 🧪 Teste tes propres questions

Modifie la phrase ci-dessous et relance la cellule. Quelques idées :
- *« Le hoodie Adidas beige en M, vous l'avez ? »* : la taille est en rupture, Léa propose-t-elle une autre taille ?
- *« Les AirPods 4 valent le coup ? Que disent les clients ? »*
- *« Qu'est-ce que vous avez en promo en ce moment ? »*
- *« Vous vendez des PlayStation 5 ? »* : le produit n'est pas au catalogue, Léa invente-t-elle ?
- *« Vous avez les Nike Air Max en 42 ? »* : Léa les confond-elle avec les Air Force 1 ?

In [36]:
ma_question = "Le hoodie Adidas beige en M, vous l'avez ?"
afficher_messages(agent_lg.invoke({"messages": [("user", ma_question)]})["messages"])

🔧 Appelle    │ verifier_stock({'produit': 'Hoodie Adidas', 'couleur': 'beige', 'taille': 'M'})
📦 Reçoit     │ hoodie adidas essentials beige : taille 'M' indisponible. Tailles en stock : S, L, XL.

💬 Répond     │ Bonjour — le hoodie Adidas Essentials beige en taille M est malheureusement indisponible.  
Ce modèle est en stock uniquement en tailles S, L et XL.  
Souhaitez-vous que je cherche la même en une autre couleur ou que je vérifie une autre taille pour vous ? — Léa


---
## 🎓 Récapitulatif

| Concept | En une phrase | Python pur | Framework |
|---|---|---|---|
| **Outil** | Une fonction Python que le LLM peut demander | `def` + fiche JSON (`decrire`) | `tool(fonction)` (LangChain) |
| **Tool calling** | Le LLM demande, ton code exécute | `tools=SCHEMAS` | `llm.bind_tools()` (LangChain) |
| **Boucle ReAct** | Réfléchir, agir, observer, recommencer | boucle `for` | `StateGraph` + `ToolNode` (LangGraph) |
| **Validation** | Vérifier les arguments du LLM avant d'agir | `executer()` + Pydantic | `tool(..., args_schema=...)` |
| **Agent** | LLM + outils + boucle | `agent_python()` | `agent_lg` |

**Les 4 idées à garder**
1. Le LLM ne connaît pas ta boutique : **les outils lui donnent accès aux vraies données**.
2. Le LLM **ne fait que demander**. Ton code exécute, donc tu gardes le contrôle.
3. Ce que le LLM envoie **se valide** (Pydantic) avant d'être exécuté, et les actions à risque demandent d'autres garde-fous.
4. Les frameworks **ne font rien de magique** : ils automatisent la boucle que tu as écrite à la main.

➡️ **Prochain notebook** : Léa va lire des documents (guides d'achat, CGV, FAQ) pour répondre à des questions plus ouvertes. C'est le **RAG**.